# AgriSense Phase 3 — PlantVillage MobileNetV2 training

Use a Kaggle GPU accelerator (T4 or P100), enable Internet for pretrained ImageNet weights when needed, and attach [PlantVillage](https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset) with **Add Input**. The checked mapping and model output order live in `ml/train.py`.

The notebook runs the repository's training script, which writes a stratified evaluation report, exports static QDQ int8 ONNX, and verifies PyTorch/ONNX Runtime predictions on five test images before installing `backend/app/ml/model.onnx` and `labels.json`. Training is capped at 23 minutes by default; data validation and export run outside that training timer.

If GitHub cloning is unavailable, upload the repository ZIP as a Kaggle input, extract it to `/kaggle/working/AgriSense`, and then run the setup cell.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path('/kaggle/working/AgriSense')
if not (REPO / 'ml' / 'train.py').is_file():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/sambitkarmakar03/AgriSense.git', str(REPO)
    ], check=True)

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Repository:', REPO)
print('Training script:', (REPO / 'ml' / 'train.py').resolve())
print('Kaggle inputs:', [p.name for p in Path('/kaggle/input').iterdir()])

In [ ]:
%pip install -q -r /kaggle/working/AgriSense/ml/requirements.txt

In [ ]:
from ml.train import resolve_class_root

INPUT_ROOT = Path('/kaggle/input')
CLASS_ROOT = resolve_class_root(INPUT_ROOT)
print('Validated PlantVillage class root:', CLASS_ROOT)
print('Source class folders:', sorted(p.name for p in CLASS_ROOT.iterdir() if p.is_dir() and ('___' in p.name)))

## Train, evaluate, quantize, and verify

The command defaults to 12 epochs, patience 3, batch size 32, and a 23-minute training budget. Adjust only if needed for the Kaggle GPU/session; report output is timestamped under `ml/reports/`. A successful cell prints the paths of the checked model and labels.

In [ ]:
command = [
    sys.executable, 'ml/train.py',
    '--dataset-root', str(CLASS_ROOT),
    '--epochs', '12',
    '--patience', '3',
    '--batch-size', '32',
    '--workers', '2',
    '--seed', '42',
    '--max-minutes', '23',
]
print('Running:', ' '.join(command))
subprocess.run(command, cwd=REPO, check=True)

## Package downloadable outputs

After successful training and parity verification, this cell creates one ZIP in `/kaggle/working`. Download it from the notebook's **Output** panel. The archive includes the model, class-label metadata, and the complete report directory for this run.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

MODEL = REPO / 'backend' / 'app' / 'ml' / 'model.onnx'
LABELS = REPO / 'backend' / 'app' / 'ml' / 'labels.json'
REPORT_ROOT = REPO / 'ml' / 'reports'
RUN_REPORT = max((p for p in REPORT_ROOT.glob('run_*') if p.is_dir()), key=lambda p: p.name)
ARCHIVE = Path('/kaggle/working/agrisense_phase3_artifacts.zip')
for artifact in (MODEL, LABELS, RUN_REPORT):
    if not artifact.exists():
        raise FileNotFoundError(f'Required training output is missing: {artifact}')
with ZipFile(ARCHIVE, 'w', compression=ZIP_DEFLATED) as bundle:
    bundle.write(MODEL, 'backend/app/ml/model.onnx')
    bundle.write(LABELS, 'backend/app/ml/labels.json')
    for report_file in sorted(RUN_REPORT.rglob('*')):
        if report_file.is_file():
            bundle.write(report_file, str(Path('ml/reports') / RUN_REPORT.name / report_file.relative_to(RUN_REPORT)))
print('Download from Kaggle Output:', ARCHIVE)
print('Model bytes:', MODEL.stat().st_size)
print('Report directory:', RUN_REPORT)